# 03. PromptTemplate과 Message

PromptTemplate을 사용하면 프롬프트를 재사용 가능한 템플릿으로 관리할 수 있습니다.

In [5]:
from pathlib import Path
from dotenv import load_dotenv

# 프로젝트 폴더와 노트북 폴더 어느 쪽에서 실행해도 같은 .env 사용
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
load_dotenv(project_root / ".env", override=True)


True

## 1. PromptTemplate - 단일 문자열 템플릿

In [6]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘") # f-string과 달리 f가 붙지 않음

# 변수 채우기
prompt = template.invoke({"topic": "RAG"}) # invoke라고 무조건 llm에 다녀오는 것이 아니라, prompt를 채우는 것임. llm에 다녀오는 것은 llm.invoke(prompt)임.
print(prompt)

text='RAG에 대해 한 문장으로 설명해줘'


## 2. ChatPromptTemplate - 역할 기반 템플릿

시스템/사용자/AI 메시지를 조합하여 템플릿을 만듭니다.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다."),
    ("human", "{question}")
])

messages = chat_prompt.invoke({
    "role": "파이썬",
    "question": "데코레이터가 뭔가요?"
})

print(messages)

messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데코레이터가 뭔가요?', additional_kwargs={}, response_metadata={})]


## 3. LLM과 연결하여 호출

In [10]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    temperature=0,
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
    use_responses_api=False,
)

messages = chat_prompt.invoke({
    "role": "머신러닝",
    "question": "과적합이 뭔가요?"
})

response = llm.invoke(messages) # llm과 통신하여 답변을 받아옴
print(response.content)

과적합(overfitting)은 **모델이 학습 데이터에는 너무 잘 맞지만, 새로운 데이터에는 잘 못 맞는 상태**를 말합니다.

예를 들어:
- 시험 문제를 통째로 외운 학생은
- 같은 문제는 잘 풀지만
- 조금만 바뀐 문제는 못 풀 수 있죠.

머신러닝도 비슷합니다.

### 과적합이 생기는 이유
- 모델이 너무 복잡할 때
- 학습 데이터가 너무 적을 때
- 노이즈까지 정답처럼 외울 때

### 과적합의 특징
- **훈련 정확도는 높다**
- 하지만 **검증/테스트 정확도는 낮다**

### 방지 방법
- 더 많은 데이터 사용
- 모델 단순화
- 정규화(regularization)
- 드롭아웃(dropout)
- 조기 종료(early stopping)
- 데이터 증강(data augmentation)

원하시면 제가 **과적합, 과소적합, 일반화**를 같이 비교해서 설명해드릴게요.


## 4. 대화 히스토리 포함 - MessagesPlaceholder

이전 대화 내용을 프롬프트에 삽입할 때 사용합니다.

In [11]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 이전 대화 기록
history = [
    HumanMessage(content="내 이름은 홍길동이야"),
    AIMessage(content="안녕하세요, 홍길동님!")
]

messages = chat_prompt.invoke({
    "history": history,
    "question": "내 이름이 뭐라고?"
})

response = llm.invoke(messages)
print(response.content)

당신의 이름은 홍길동입니다.


In [12]:
# 현재 대화를 history에 추가
history.append(HumanMessage(content="내 이름이 뭐라고?"))
history.append(AIMessage(content=response.content))
history

[HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='당신의 이름은 홍길동입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 5. 다중 변수 템플릿 예제

In [13]:
template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 번역가입니다."),
    ("human", "다음 문장을 번역해줘: {sentence}")
])

messages = template.invoke({
    "language": "영어",
    "sentence": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

The weather is really nice today.


## 문제 해결

In [24]:
template = ChatPromptTemplate.from_messages([
    ("system", """당신은 {sorted_language}에서 {target_language}로 번역하는 번역가입니다.
      출력을 할 때 {sorted_language}로 번역된 문장과 {target_language}로 번역된 문장을 모두 보여주세요.
     
      예시:
      입력(한국어): "오늘 날씨가 정말 좋네요.
      출력(영어): The weather is really nice today. 
     """),
    ("human", "다음 문장을 번역해줘: {text}")
])

messages = template.invoke({
    "sorted_language": "한국어",
    "target_language": "영어",
    "text": "당신은 아름답다."
})

response = llm.invoke(messages)
print(response.content)

한국어: 당신은 아름답다.
영어: You are beautiful.
